# 피트니스 데이터 선형회귀 분석

1. 선형회귀 기초 이론 (Linear regression)
1. 당뇨 데이터셋
1. 피트니스 데이터셋 (소쿠리 대회)
1. 다항회귀 (Polinomial regression)

### 참고자료
- [파이썬 3 표준 문서](https://docs.python.org/3/index.html)
- [Scikit learn Linear regression](https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html)
- [Diabetes dataset](https://www4.stat.ncsu.edu/~boos/var.select/diabetes.html)
- [TensorFlow Linear regression](https://www.tensorflow.org/tutorials/keras/regression)

### 당뇨병 (Diabetes) 데이터셋

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas.plotting import scatter_matrix
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [ ]:
diabetes = datasets.load_diabetes()
print(diabetes.DESCR)

In [ ]:
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df

In [ ]:
df['y'] = diabetes.target
df

In [ ]:
scatter_matrix(df[df.columns],
               c=df['y'],
               alpha=0.5,
               figsize=(7, 7),)
print('') # Slient

In [ ]:
features = ['age']
#features = ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
X = df[features]
y = df['y']

In [ ]:
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)
print(f'학습에 사용할 피처 크기: {X_train.shape}')
print(f'예측에 사용할 피처 크기: {X_test.shape}')

In [ ]:
regr = linear_model.LinearRegression()
regr.fit(X_train, y_train)
regr

In [ ]:
y_pred = regr.predict(X_test)

print("Coefficients: \n", regr.coef_)
print("Mean squared error: %.2f" % mean_squared_error(y_test, y_pred))
print("Coefficient of determination: %.2f" % r2_score(y_test, y_pred))

In [ ]:
fig = plt.figure(figsize= (8, 6))
ax = fig.add_subplot()
ax.scatter(X_test, y_test, color='black')
ax.plot(X_test, y_pred, color='blue', linewidth=3)

ax.set_title(f'Diabetes progression by {features}', fontsize='x-large')
ax.set_xlabel(f'{features}', fontsize='large')
ax.set_ylabel('Diabetes progression', fontsize='large')

In [ ]:
regr.coef_

In [ ]:
regr.intercept_

### 피트니스 데이터 회귀 분석

In [ ]:
import os
import datetime
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
!tar -xvf sokulee.tar > /dev/null

In [ ]:
file_list = []
ext = '.json'

queue = [os.path.abspath(os.path.expanduser('./sokulee')),
        ]

counter = 0
while queue:
    cursor = queue.pop()
    counter = counter + 1
    with os.scandir(cursor) as it:
        for entry in it:
            if entry.is_dir():
                queue.append(entry.path)
            elif entry.is_file() and entry.path.endswith(ext):
                file_list.append(entry.path)

print(f'{counter}개의 디렉터리에서 {len(file_list)}개 {ext} 파일 발견')
for path in file_list[:3]:
    print(path)

In [ ]:
df_steps = pd.DataFrame()
counter1 = 0
counter2 = 0
for path in file_list:
    if 'steps' not in path:
        continue
    counter1 = counter1 + 1
    steps = []
    with open(path, 'r') as f:
        data = json.load(f)
        user = os.path.basename(path).split('_')[0]
        if 'activities-steps' not in data:
            print(f'처리 불가: {path}')
            continue
        day = data['activities-steps'][0]['dateTime']
        for row in data['activities-steps-intraday']['dataset']:
            datetime_str = f'{day}T{row["time"]}+09:00'
            datetime_iso = datetime.datetime.fromisoformat(datetime_str)
            value = row['value']
            steps.append({'user': user,
                          'datetime': datetime_iso,
                          'steps': value})
    counter2 = counter2 + 1
    new_df_steps = pd.DataFrame(steps)
    df_steps = pd.concat([df_steps, new_df_steps], ignore_index=True)
print(f'{counter1}개 파일 중 {counter2}개 입력됨')
df_steps

In [ ]:
print(f'기간: {min(df_steps["datetime"])} ~ {max(df_steps["datetime"])}')
print(f'참여 인원: {len(df_steps["user"].unique())}')

In [ ]:
daily_steps = df_steps.groupby(['user', df_steps['datetime'].dt.strftime('%Y-%m-%d')])['steps'].sum().reset_index()
daily_steps.rename(columns={'datetime': 'date'}, inplace=True)
daily_steps.set_index(['user', 'date'], inplace=True)
daily_steps

In [ ]:
df_steps['user'].unique()

In [ ]:
daily_steps.loc[('A035', '2016-05-02')]

In [ ]:
user_mean_steps = daily_steps.groupby('user')['steps'].mean().reset_index()
user_mean_steps

In [ ]:
fig = plt.figure(figsize= (8, 6))
ax = fig.add_subplot()
h = ax.hist(user_mean_steps['steps'])
ax.tick_params(labelsize='large')
ax.set_xlabel('Daily steps', fontsize='large')
ax.set_ylabel('Count', fontsize='large')
_ = ax.set_title('Daily steps by user histogram', fontsize='x-large')

In [ ]:
fig = plt.figure(figsize= (8, 6))
ax = fig.add_subplot()
cdf = ax.hist(daily_steps['steps'], bins=max(daily_steps.values)[0]+1, cumulative=True, histtype='step', density=True)
ax.axvline(x=10000, color='red', linestyle='--')
ax.set_xlim((-200, 20200))
ax.set_ylim((-0.01, 1.01))
ax.tick_params(labelsize='large')
ax.set_xlabel('Daily steps', fontsize='large')
ax.set_ylabel('CDF', fontsize='large')
_ = ax.set_title('Daily steps CDF', fontsize='x-large')

In [ ]:
goal = daily_steps[daily_steps['steps'] >= 10000].groupby('date').size().reset_index(name='count')
goal.head()

In [ ]:
goal.loc[goal['count'] == max(goal['count'])]

In [ ]:
fig = plt.figure(figsize= (8, 6))
ax = fig.add_subplot()
ax.plot(goal['count'], color='green')
ax.axhline(y=len(df_steps["user"].unique())//2, color='red', linestyle='--')
ax.tick_params(axis='x', labelrotation=90, labelsize='medium')
ax.tick_params(axis='y', labelsize='large')
ax.set_xlabel('Day', fontsize='large')
ax.set_ylabel('Count', fontsize='large')
_ = ax.set_title('Daily goal achievement count', fontsize='x-large')